In [4]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import spacy
import re
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
bbc_data = pd.read_csv("bbc_news.csv")

In [27]:
#extracting titles from the CSV and creating a new pandas dataframe
titles = pd.DataFrame(bbc_data['title'])
#cleaning up the data
#converting to lowercase
titles['lowercase'] = titles['title'].str.lower()

#removing stop words
en_stopwords = stopwords.words('english')
titles['no_stopwords'] = titles['lowercase'].apply(lambda x: ' '.join([word for word in x.split() if word not in en_stopwords ]))

#removing punctuation
titles['no_stopwords_no_punct'] =  titles.apply(lambda x: re.sub("([^\w\s])", "", x["no_stopwords"]), axis=1)

# tokenize
titles['tokens_raw'] = titles.apply(lambda x: word_tokenize(x['title']), axis=1)
titles['tokens_clean'] = titles.apply(lambda x: word_tokenize(x['no_stopwords_no_punct']), axis=1)

# lemmatizing 
lemmatizer = WordNetLemmatizer()
titles["tokens_clean_lemmatized"] = titles["tokens_clean"].apply(lambda tokens: [lemmatizer.lemmatize(token) for token in tokens])
titles.head()

tokens_raw_list = sum(titles["tokens_raw"], [])
tokens_clean_list = sum(titles["tokens_clean_lemmatized"], [])

In [38]:
#POS tagging
nlp = spacy.load("en_core_web_sm")
spacy_doc = nlp(' '.join(tokens_raw_list))
pos_df = pd.DataFrame(columns=['token','pos_tag'])
for token in spacy_doc:
    pos_df = pd.concat([pos_df, pd.DataFrame.from_records([{'token':token.text, 'pos_tag':token.pos_}])], ignore_index=True)
# token frequency count
pos_df_counts = pos_df.groupby(['token','pos_tag']).size().reset_index(name='counts').sort_values(by='counts', ascending=False)
pos_df_counts.head(10)

,token,pos_tag,counts
95,:,PUNCT,543
8,',PUNCT,300
2897,in,ADP,187
4082,to,PART,175
3268,of,ADP,172
22,-,PUNCT,166
4043,the,DET,163
1856,and,CCONJ,147
15,'s,PART,143
97,?,PUNCT,130


In [41]:
#filter nouns.
noun = pos_df_counts[pos_df_counts.pos_tag == 'NOUN'] #can filter for VERBS, ADJ
noun

,token,pos_tag,counts
4267,war,NOUN,35
3552,record,NOUN,15
3416,police,NOUN,14
4316,win,NOUN,14
4356,year,NOUN,14
...,...,...,...
4363,zone,NOUN,1
4342,worker,NOUN,1
4347,worsens,NOUN,1
4337,wonder,NOUN,1


In [50]:
#NER Tagging
ner_df =  pd.DataFrame(columns =['token', 'ner_tag'])
for token in spacy_doc.ents:
    if pd.isna(token.label_) is False:
        ner_df = pd.concat([ner_df, pd.DataFrame.from_records([{'token': token.text, 'ner_tag': token.label_}])], ignore_index=True)
ner_df.head()

,token,ner_tag
0,Liz Truss,PERSON
1,UK,GPE
2,Rationing,PRODUCT
3,superyachts,CARDINAL
4,Russian,NORP


In [53]:
ner_df_counts = ner_df.groupby(['token', 'ner_tag']).size().reset_index(name='counts').sort_values(by='counts', ascending=False)
ner_df_counts.head()

,token,ner_tag,counts
965,Ukraine,GPE,47
955,UK,GPE,36
329,England,GPE,32
819,Russian,NORP,20
957,US,GPE,19
